In [ ]:
!pip install pandas numpy openai sentence-transformers datasets transformers[torch] accelerate>=1.1.0 seaborn hdbscan umap-learn nltk plotly optuna BERTopic "git+https://github.com/FelSiq/DBCV"

In [2]:
%load_ext cuml.accel

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML: Installed accelerator for sklearn.
cuML: Installed accelerator for umap.
cuML: Installed accelerator for hdbscan.
cuML: Successfully initialized accelerator.


In [2]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI
import random
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import safetensors
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
import textwrap
import gc
import re
import requests
import html
import re
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.set_option('display.max_columns', None)

# Data Preprocessing

1. Load the full data extracted from Neo4j
2. Select only bill types that are Motion or Parliamentary Initiative
3. Format the bill number compliant with the Affair_ID to later fetch bill metadata 
4. Select the top 25 multilingual bills by # of paragraphs
5. Use NLI to classify whether a paragraph is procedural or not, and filter out procedural paragraphs
6. Aggregate the (non-procedural) paragraphs at the speech level

In [4]:
def normalize_swiss_bill(val):
    # Handle missing/NaN values safely
    if pd.isna(val) or val == 'nan':
        return np.nan

    # Ensure it is processed as a string representation of the float
    val_str = str(val).strip()
    if '.' not in val_str:
        return val_str

    left, right = val_str.split('.')

    # 1. Left-pad the year to 2 digits
    normalized_left = left.zfill(2)

    # 2. Right-pad the sequence to at least 3 digits
    normalized_right = right.ljust(3, '0')

    return f"{normalized_left}.{normalized_right}"

In [5]:
def fetch_bill_metadata(affair_id: str):
    if pd.isna(affair_id):
        return None

    url = f"https://ws-old.parlament.ch/affairs/{affair_id}?lang=en&format=json"
    headers = {"Accept": "application/json"}

    response = requests.get(url, headers=headers)

    if response.ok:
        data = response.json()

        # --- NEW: Extracting from the exact JSON schema provided ---
        extracted_keywords = []

        # Loop through the descriptors array to extract names
        descriptors = data.get('descriptors', [])
        if isinstance(descriptors, list):
            for item in descriptors:
                name = item.get('name')
                if name:
                    extracted_keywords.append(name.strip())

        # Fallback to additionalIndexing if descriptors are missing
        if not extracted_keywords and "additionalIndexing" in data:
            raw_indexing = data.get("additionalIndexing", "")
            # Filter out the pure structural numbers (like '24', '2846')
            tags = [t.strip() for t in raw_indexing.split(";") if not t.strip().isdigit()]
            extracted_keywords.extend(tags)

        # Deduplicate while keeping sequence order
        keywords_str = ", ".join(list(dict.fromkeys(extracted_keywords)))
        # -----------------------------------------------------------

        texts = data.get('texts', [])
        bill_metadata = {
            "title": data.get("title"),
            "keywords": keywords_str,  # Populates: "Wohneigentum, Grundstückgewinnsteuer, ..."
            "ideological_summary": "",
            "statutory_text": ""
        }

        for text in texts:
            text_type_name = text.get("type", {}).get("name")
            raw_value = text.get("value", "")

            clean_value = re.sub(r'<[^>]+>', ' ', html.unescape(raw_value))
            clean_value = ' '.join(clean_value.split())

            if text_type_name in ["Développement", "Bedeutung und Begründung", "Development"]:
                bill_metadata["ideological_summary"] = clean_value
            elif text_type_name in ["Texte déposé", "Eingereichter Text", "Submitted text"]:
                bill_metadata["statutory_text"] = clean_value

        return bill_metadata

    print(f"Error fetching data. Status code: {response.status_code}")
    return None

In [6]:
def business_number_to_affair_id(normalized_str):
    if pd.isna(normalized_str) or normalized_str in ['nan', 'NaN']:
        return np.nan

    left, right = normalized_str.split('.')
    year_short = int(left)
    sequence = int(right)

    # 1. Deduce the 4-digit century year
    # Adjust the cutoff threshold dynamically depending on the data age
    if year_short <= 26:
        year_full = 2000 + year_short
    else:
        year_full = 1900 + year_short

    # 2. Left-pad the integer sequence to create the 8-digit block
    # Equivalent to: (year_full * 10000) + sequence
    affair_id = f"{year_full}{str(sequence).zfill(4)}"
    return affair_id

In [7]:
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline


# --------------------------------------------------
# Fast heuristic filter
# --------------------------------------------------

def has_heuristic_procedural_patterns(text):
    text = str(text).strip()

    if text == "":
        return True

    patterns = [
        r"^\[vs\]$",
        r"^angenommen\s*-\s*adopté$",
        r"^adopté\s*-\s*angenommen$",
        r"^das wort hat$",
        r"^le parole est à$",
    ]

    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE):
            return True

    if len(text.split()) < 3:
        return True

    return False

In [8]:
# --------------------------------------------------
# Zero-shot discourse classifier
# --------------------------------------------------

DISCOURSE_LABELS = [
    "Procedural parliamentary remark",
    "Legislative background or factual context",
    "Policy evidence or justification",
    "Policy argument",
    "Explicit policy recommendation or stance",
]

In [9]:
def classify_discourse_paragraphs(
    df,
    text_col="TEXT_SPAN",
    batch_size=64,
    model_name="MoritzLaurer/bge-m3-zeroshot-v2.0",
):

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

    df = df.copy()

    # obvious procedural
    heuristic_mask = df[text_col].apply(has_heuristic_procedural_patterns)

    df["discourse_label"] = "Procedural parliamentary remark"
    df["confidence"] = 1.0

    label_cols = {
        label: []
        for label in DISCOURSE_LABELS
    }

    to_predict = df.loc[~heuristic_mask]

    texts = to_predict[text_col].fillna("").astype(str).tolist()

    outputs = classifier(
        texts,
        candidate_labels=DISCOURSE_LABELS,
        multi_label=False,
        batch_size=batch_size,
    )

    for row_idx, out in zip(to_predict.index, outputs):

        best_label = out["labels"][0]
        best_score = out["scores"][0]

        df.loc[row_idx, "discourse_label"] = best_label
        df.loc[row_idx, "confidence"] = best_score

        score_dict = dict(zip(out["labels"], out["scores"]))

        for label in DISCOURSE_LABELS:
            value = score_dict.get(label, 0.0)
            label_cols[label].append((row_idx, value))

    for label in DISCOURSE_LABELS:

        column = (
            label.lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

        df[column] = 0.0

        for idx, score in label_cols[label]:
            df.loc[idx, column] = score

    return df

In [10]:
import pandas as pd
df = pd.read_csv("data/speeches_with_text.csv")

In [11]:
df

,Speech_ID,speaker_uid,speaker_first_name,speaker_last_name,speaker_fullname,party,party_name,gender,source_name,source_abbrev,meeting_uid,meeting_number,meeting_datetime,meeting_date,session_name,year,legislative_period_uid,Debate_ID,Bill_Number,Bill_Title_DE,Bill_Title_FR,Bill_Labels,Discussion_ID,Partition_ID,Paragraph_ID,name,role,language_of_text,TEXT_SPAN
0,81893,453.0,André,Bugnon,Bugnon André,SVP,Schweizerische Volkspartei,m,Online DB,ODB,941.0,12.0,2007-12-20T08:00:00.000000000,2007-12-20,Winter session,2007.0,48.0,06_405.5,6.405,Verdoppelung der Anzahl Durchdiener,Doublement de la proportion de militaires en s...,Bill|Parliamentary Initiative,12870.0,12870.1,1,Bugnon André,President,FR,La commission propose de ne pas donner suite à...
1,81893,453.0,André,Bugnon,Bugnon André,SVP,Schweizerische Volkspartei,m,Online DB,ODB,941.0,12.0,2007-12-20T08:00:00.000000000,2007-12-20,Winter session,2007.0,48.0,06_405.5,6.405,Verdoppelung der Anzahl Durchdiener,Doublement de la proportion de militaires en s...,Bill|Parliamentary Initiative,12870.0,12870.1,2,Bugnon André,President,FR,[VS]
2,81893,453.0,André,Bugnon,Bugnon André,SVP,Schweizerische Volkspartei,m,Online DB,ODB,941.0,12.0,2007-12-20T08:00:00.000000000,2007-12-20,Winter session,2007.0,48.0,06_405.5,6.405,Verdoppelung der Anzahl Durchdiener,Doublement de la proportion de militaires en s...,Bill|Parliamentary Initiative,12870.0,12870.1,3,Bugnon André,President,FR,Angenommen - Adopté
3,81894,390.0,Hans,Widmer,Widmer Hans,SP,Sozialdemokratische Partei der Schweiz,m,Online DB,ODB,941.0,12.0,2007-12-20T08:00:00.000000000,2007-12-20,Winter session,2007.0,48.0,06_405.5,6.405,Verdoppelung der Anzahl Durchdiener,Doublement de la proportion de militaires en s...,Bill|Parliamentary Initiative,12870.0,12870.1,1,Widmer Hans,For the committee,DE,"Ich möchte Sie nur nochmals bitten, das Result..."
4,81894,390.0,Hans,Widmer,Widmer Hans,SP,Sozialdemokratische Partei der Schweiz,m,Online DB,ODB,941.0,12.0,2007-12-20T08:00:00.000000000,2007-12-20,Winter session,2007.0,48.0,06_405.5,6.405,Verdoppelung der Anzahl Durchdiener,Doublement de la proportion de militaires en s...,Bill|Parliamentary Initiative,12870.0,12870.1,2,Widmer Hans,For the committee,DE,"In diesem Sinne bitte ich Sie nochmals, den An..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
666686,288682,3912.0,Christian,Wasserfallen,Wasserfallen Christian,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,4518.0,9.0,2021-09-23T08:00:00.000000000,2021-09-23,Fall session,2021.0,51.0,21_049.1,21.049,Gentechnikgesetz. Änderung,Loi sur le génie génétique. Modification,Bill|Federal Council Dispatch,54270.0,54270.1,8,Wasserfallen Christian,NaN,DE,Noch etwas zur Technologie: Es grenzt an Absur...
666687,288682,3912.0,Christian,Wasserfallen,Wasserfallen Christian,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,4518.0,9.0,2021-09-23T08:00:00.000000000,2021-09-23,Fall session,2021.0,51.0,21_049.1,21.049,Gentechnikgesetz. Änderung,Loi sur le génie génétique. Modification,Bill|Federal Council Dispatch,54270.0,54270.1,9,Wasserfallen Christian,NaN,DE,Die jüngst erschienene Befragung zeigt deutlic...
666688,288682,3912.0,Christian,Wasserfallen,Wasserfallen Christian,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,4518.0,9.0,2021-09-23T08:00:00.000000000,2021-09-23,Fall session,2021.0,51.0,21_049.1,21.049,Gentechnikgesetz. Änderung,Loi sur le génie génétique. Modification,Bill|Federal Council Dispatch,54270.0,54270.1,10,Wasserfallen Christian,NaN,DE,"Lange Forschung und Rede, kurzer Sinn: In eine..."
666689,288682,3912.0,Christian,Wasserfallen,Wasserfallen Christian,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,4518.0,9.0,2021-09-23T08:00:00.000000000,2021-09-23,Fall session,2021.0,51.0,21_049.1,21.049,Gentechnikgesetz. Änderung,Loi sur le génie génétique. Modification,Bill|Federal Council Dispatch,54270.0,54270.1,11,Wasserfallen Christian,NaN,DE,"Ich bitte Sie deshalb, wenigstens

In [12]:
# Select Motion and Parliamentary Initiative bills
df = df[df['Bill_Labels'].isin(['Bill|Motion', 'Bill|Parliamentary Initiative'])]

In [13]:
# Apply normalization helpers to the raw dataset first
df['Bill_Number'] = df['Bill_Number'].apply(normalize_swiss_bill)

# Drop rows where Bill_Number or TEXT_SPAN is missing
df = df.dropna(subset=['Bill_Number', 'TEXT_SPAN', 'language_of_text'])
df['lang'] = df['language_of_text'].str.upper().str.strip()

# Generate Affair_ID for the API calls
df['Affair_ID'] = df['Bill_Number'].apply(business_number_to_affair_id)

/tmp/ipykernel_2377/1588943188.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Bill_Number'] = df['Bill_Number'].apply(normalize_swiss_bill)


In [14]:
# How many bills?
df['Bill_Number'].nunique()

5386

In [15]:
N_TARGET_BILLS = 20
dense_multilingual_bills = df.groupby('Bill_Number').size().sort_values(ascending=False).head(N_TARGET_BILLS).index.tolist()

print("Example selected bills:", dense_multilingual_bills)

Example selected bills: ['10.443', '21.501', '19.401', '19.475', '09.528', '19.443', '04.476', '05.453', '19.400', '08.445', '21.502', '07.400', '06.441', '13.468', '17.400', '12.400', '07.492', '16.457', '04.430', '10.467']


In [16]:
# Filter our main dataframe to only contain these dense bills
df_filtered = df[df['Bill_Number'].isin(dense_multilingual_bills)].copy().reset_index(drop=True)

In [17]:
import requests

bill_metadata_cache = {}
unique_affairs = df_filtered['Affair_ID'].dropna().unique()

In [18]:
unique_affairs[0]

'20040476'

In [19]:
fetch_bill_metadata(unique_affairs[0])

{'title': "Protection de la population et de l'économie contre le tabagisme passif",
 'keywords': 'Tabakkonsum, Nikotinsucht, Gesundheitszustand, Gaststättengewerbe, Gastronomieberuf, öffentliches Gebäude, Arbeitsstätte, Schule, Spital, öffentlicher Verkehr',
 'ideological_summary': "Situation de départ et action nécessaire Chaque jour, même sans avoir jamais fumé, quelqu'un meurt en Suisse à cause du tabagisme. Au cours d'une année, ils sont des milliers à tomber malades, dont beaucoup sont gravement atteints. Tous les âges sont touchés. Les coûts pour la santé publique et pour notre économie sont considérables, les souffrances sont grandes et une forte majorité de la population suisse voit sa liberté quotidienne sensiblement réduite. L'industrie du tabac reconnaît, par ailleurs, que le tabagisme vis-à-vis des tiers, dit tabagisme passif, rend malade et tue. Elle aussi se prononce en faveur de mesures contre le tabagisme passif, qui est la pire source de pollution des espaces intérieu

In [20]:
for affair_id in tqdm(unique_affairs, desc="Fetching Bill Metadata"):
    meta = fetch_bill_metadata(affair_id)
    if meta:
        bill_metadata_cache[affair_id] = meta

Fetching Bill Metadata: 100% 20/20 [00:03<00:00,  5.23it/s]


In [21]:
bill_metadata_cache

{'20040476': {'title': "Protection de la population et de l'économie contre le tabagisme passif",
  'keywords': 'Tabakkonsum, Nikotinsucht, Gesundheitszustand, Gaststättengewerbe, Gastronomieberuf, öffentliches Gebäude, Arbeitsstätte, Schule, Spital, öffentlicher Verkehr',
  'ideological_summary': "Situation de départ et action nécessaire Chaque jour, même sans avoir jamais fumé, quelqu'un meurt en Suisse à cause du tabagisme. Au cours d'une année, ils sont des milliers à tomber malades, dont beaucoup sont gravement atteints. Tous les âges sont touchés. Les coûts pour la santé publique et pour notre économie sont considérables, les souffrances sont grandes et une forte majorité de la population suisse voit sa liberté quotidienne sensiblement réduite. L'industrie du tabac reconnaît, par ailleurs, que le tabagisme vis-à-vis des tiers, dit tabagisme passif, rend malade et tue. Elle aussi se prononce en faveur de mesures contre le tabagisme passif, qui est la pire source de pollution des e

In [22]:
bill_metadata_df = pd.json_normalize(df_filtered['Affair_ID'].map(bill_metadata_cache))

In [23]:
df_complete = pd.concat([df_filtered.reset_index(drop=True), bill_metadata_df.reset_index(drop=True)], axis=1)

In [24]:
df_complete

,Speech_ID,speaker_uid,speaker_first_name,speaker_last_name,speaker_fullname,party,party_name,gender,source_name,source_abbrev,meeting_uid,meeting_number,meeting_datetime,meeting_date,session_name,year,legislative_period_uid,Debate_ID,Bill_Number,Bill_Title_DE,Bill_Title_FR,Bill_Labels,Discussion_ID,Partition_ID,Paragraph_ID,name,role,language_of_text,TEXT_SPAN,lang,Affair_ID,title,keywords,ideological_summary,statutory_text
0,83161,1149.0,Urs,Schwaller,Schwaller Urs,CVP,Christlichdemokratische Volkspartei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schwaller Urs,For the committee,DE,Ein einziger Satz: Wir waren hier der Auffassu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
1,83162,425.0,Rolf,Schweiger,Schweiger Rolf,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schweiger Rolf,NaN,DE,Ich begründe meinen Antrag ganz kurz: Gesetzli...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
2,83162,425.0,Rolf,Schweiger,Schweiger Rolf,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,2,Schweiger Rolf,NaN,DE,Nun ist die Sache insofern eine Blackbox-Angel...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
3,83162,425.0,Rolf,Schweiger,Schweiger Rolf,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,3,Schweiger Rolf,NaN,DE,[PAGE 39],DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
4,83163,276.0,Christoffel,Brändli,Brändli Christoffel,SVP,Schweizerische Volkspartei,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Brändli Christoffel,President,DE,Die Kommission beantragt aufgrund der Abstimmu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14934,288162,4152.0,Hans,Wicki,Wicki Hans,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,4514.0,6.0,2021-09-21T08:05:00.000000000,2021-09-21,Fall session,2021.0,51.0,17_400.3,17.400,Systemwechsel bei der Wohneigentumsbesteuerung,Imposition du logement. Changement de système,Bill|Parliamentary Initiative,541

In [37]:
aggregation = {
    k: "first" for k in df_complete.columns if k != "Speech_ID"
}
aggregation["TEXT_SPAN"] = lambda x: " ".join(x.astype(str))

df_speech_level = (
    df_complete
    .groupby("Speech_ID")
    .agg(aggregation)
    .reset_index()
)

In [41]:
df_speech_level

,Speech_ID,speaker_uid,speaker_first_name,speaker_last_name,speaker_fullname,party,party_name,gender,source_name,source_abbrev,meeting_uid,meeting_number,meeting_datetime,meeting_date,session_name,year,legislative_period_uid,Debate_ID,Bill_Number,Bill_Title_DE,Bill_Title_FR,Bill_Labels,Discussion_ID,Partition_ID,Paragraph_ID,name,role,language_of_text,TEXT_SPAN,lang,Affair_ID,title,keywords,ideological_summary,statutory_text
0,83161,1149.0,Urs,Schwaller,Schwaller Urs,CVP,Christlichdemokratische Volkspartei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schwaller Urs,For the committee,DE,Ein einziger Satz: Wir waren hier der Auffassu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
1,83162,425.0,Rolf,Schweiger,Schweiger Rolf,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schweiger Rolf,None,DE,Ich begründe meinen Antrag ganz kurz: Gesetzli...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
2,83163,276.0,Christoffel,Brändli,Brändli Christoffel,SVP,Schweizerische Volkspartei,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Brändli Christoffel,President,DE,Die Kommission beantragt aufgrund der Abstimmu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
3,83164,1149.0,Urs,Schwaller,Schwaller Urs,CVP,Christlichdemokratische Volkspartei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schwaller Urs,For the committee,DE,Ich beantrage Ihnen hier die Streichung von Ab...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
4,83166,50.0,Pascal,Couchepin,Couchepin Pascal,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,04.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Couchepin Pascal,President of the federal council,FR,"Si vous biffez cette disposition, vous risquez...",FR,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3136,324986,3999.0,Katharina,Prelicz-Huber,Prelicz-Huber Katharina,Grüne,Grüne Partei,f,Online DB,ODB,4803.0,4.0,2023-09-13T15:00:00.000000000,2023-09-13,Fall session,2023.0,51.0,09_528.5,09.528,Fin

In [42]:
df_speech_level.to_csv("filtered_speech_level_data.csv", index=False)